In [ ]:
import pandas as pd
import os
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

data_path = '../../datasets/alexa-top-1m/top-1m.csv'
chunks_folder = '../../datasets/alexa-top-1m/chunks'
chunks_2nd_attempt_folder = '../../datasets/alexa-top-1m/chunks/2nd_attempt'
chunks_with_vpn_folder = '../../datasets/alexa-top-1m/chunks/2nd_attempt/with_vpn'

df = pd.read_csv(data_path, index_col=0).reset_index(drop=True)

df

,url
0,google.com
1,youtube.com
2,facebook.com
3,baidu.com
4,wikipedia.org
...,...
999995,sibf.org
999996,bukapintu.co
999997,klatovynet.cz
999998,elconquistadorfm.cl


In [2]:
df.describe()

,url
count,1000000
unique,1000000
top,google.com
freq,1


In [3]:
# splitting the data into chunks (50 chunks) (the process takes too long, even for a chunk)

chunk_size = len(df) // 50

chunks =  [df.iloc[i:i+chunk_size] for i in range(0, len(df), chunk_size)]

In [4]:
print(f'number of chunks {len(chunks)}')
for chunk in chunks:
  print(chunk.shape)

number of chunks 50
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)
(20000, 1)


In [ ]:
domain = "google.com.eg"
response = requests.get(
    f"https://{domain}",
    allow_redirects=True,
    headers={"User-Agent": "Mozilla/5.0"}
)
print("Final URL:", response.url)
print("Resolved domain:", urlparse(response.url).netloc)

Final URL: https://www.google.com/
Resolved domain: www.google.com


In [ ]:
def resolve_url(bare_domain, timeout=10):
  """
  Takes a bare domain, eg: google.com, tries to send a request to that domain with https then http scheme
  The result will be a resolved domain, eg: https://www.google.com/
  """
  try:
    response = requests.get(
      f"https://{bare_domain}",
      timeout=timeout,
      allow_redirects=True,
      headers={"User-Agent": "Mozilla/5.0"}
    )
    return response.url
  except requests.exceptions.SSLError:
    try:
      response = requests.get(
        f"http://{bare_domain}",
        timeout=timeout,
        allow_redirects=True,
        headers={"User-Agent": "Mozilla/5.0"}
      )
      return response.url
    except:
      return None
  except:
    return None

In [ ]:
def process_chunk(chunk, sub_chunk_size=None, max_workers=100):
  """
  Process a single chunk of URLs concurrently with threads.
  If sub_chunk_size is specified then the chunk will be farther splitted into sub chunks and processed
  """
  if sub_chunk_size:
    sub_chunks = [chunk.iloc[i:i+sub_chunk_size] for i in range(0, len(chunk), sub_chunk_size)]
    results = []
    for sub_chunk in sub_chunks:
      resolved_urls = 0
      sub_results = [None] * len(sub_chunk)
      with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(resolve_url, sub_chunk.iloc[i]): i for i in range(len(sub_chunk))}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Resolving sub chunk"):
          idx = futures[future]
          sub_results[idx] = future.result()
          resolved_urls += future.result() is not None
      print(f'{resolved_urls} url resolved')
      results.extend(sub_results)
    return results
  else:
    resolved_urls = 0
    results = [None] * len(chunk)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
      futures = {executor.submit(resolve_url, chunk.iloc[i]): i for i in range(len(chunk))}
      for future in tqdm(as_completed(futures), total=len(futures), desc="Resolving chunk"):
        idx = futures[future]
        results[idx] = future.result()
        resolved_urls += future.result() is not None
    print(f'{resolved_urls} url resolved')
    return results

In [ ]:
def resolve_df_urls(df, chunks_folder, chunk_size=5000, sub_chunk_size=None, max_workers=100):
  """
  Resolves all URLs in a DataFrame in chunks with threading.
  Returns the DataFrame with a new column 'resolved_url'.
  """    
  chunks = [df.iloc[i:i+chunk_size] for i in range(0, len(df), chunk_size)]

  processed_chunks = []
  for file in os.listdir(chunks_folder):
    if file.endswith('.csv'):
      processed_chunks.append(int(file.split('.')[0].split('_')[1]))

  for i, chunk in enumerate(chunks):
    if i in processed_chunks:
      continue
    print(f"Processing chunk {i + 1}/{len(chunks)} ({len(chunk)} URLs)")
    chunk_results = process_chunk(chunk['url'], sub_chunk_size=sub_chunk_size, max_workers=max_workers)
    chunk['resolved_url'] = chunk_results
    chunk.to_csv(f'{chunks_folder}/chunk_{i}.csv')

In [ ]:
resolve_df_urls(df, chunks_folder, chunk_size=20000, sub_chunk_size=5000, max_workers=100)

In [ ]:
resolved_chunks = []

for file in os.listdir(chunks_folder):
  if file.endswith('.csv'):
    resolved_chunks.append(pd.read_csv(f'{chunks_folder}/{file}', index_col=0).reset_index(drop=True))

resolved_df = pd.concat(resolved_chunks)

resolved_df

,url,resolved_url
0,buenosairescompras.gob.ar,https://buenosairescompras.gob.ar/
1,furuitang.tmall.com,NaN
2,zhujinwang.com,NaN
3,aeipro.com,NaN
4,technokarak.com,https://technokarak.com/
...,...,...
19995,lierse.com,https://www.lierse.com/
19996,poporo.ne.jp,NaN
19997,scuolasicurezza.it,NaN
19998,km43yy.com,NaN


In [12]:
# resolved VS unresolved URLs
total_rows = resolved_df.shape[0]
f'{(~resolved_df['resolved_url'].isna()).sum()} of {total_rows} resolved VS {resolved_df['resolved_url'].isna().sum()} of {total_rows} unresolved'

'628354 of 1000000 resolved VS 371646 of 1000000 unresolved'

In [13]:
# resolved VS unresolved URLs percentage
f'{(round((~resolved_df['resolved_url'].isna()).sum() / total_rows * 100))}% of {total_rows} resolved VS {round(resolved_df['resolved_url'].isna().sum() / total_rows * 100)}% of {total_rows} unresolved'

'63% of 1000000 resolved VS 37% of 1000000 unresolved'

In [14]:
unresolved_chunks = []

for chunk in resolved_chunks:
  unresolved_chunks.append(
    chunk[chunk['resolved_url'].isna()]
  )

In [15]:
unresolved_df = pd.concat(unresolved_chunks).reset_index(drop=True)

unresolved_df

,url,resolved_url
0,furuitang.tmall.com,NaN
1,zhujinwang.com,NaN
2,aeipro.com,NaN
3,automotivemileposts.com,NaN
4,bta.com,NaN
...,...,...
371641,ignition.co.za,NaN
371642,poporo.ne.jp,NaN
371643,scuolasicurezza.it,NaN
371644,km43yy.com,NaN


In [16]:
for i in range(2, len(unresolved_df)):
  if len(unresolved_df) % i == 0:
    print(i)

2
3
6
9
11
18
22
33
66
99
198
1877
3754
5631
11262
16893
20647
33786
41294
61941
123882
185823


In [17]:
len(unresolved_df) / 22

16893.0

In [18]:
for i in range(2, 16893):
  if 16893 % i == 0:
    print(i)

3
9
1877
5631


In [19]:
16893 / 3

5631.0

In [ ]:
# second attempt for unresolved urls

chunk_size = len(unresolved_df) // 22
sub_chunk_size = chunk_size // 3

resolve_df_urls(unresolved_df, chunks_2nd_attempt_folder, chunk_size=chunk_size, sub_chunk_size=sub_chunk_size, max_workers=100)

Processing chunk 15/22 (16893 URLs)


Resolving sub chunk: 100%|██████████| 5631/5631 [16:19<00:00,  5.75it/s] 


892 url resolved


Resolving sub chunk: 100%|██████████| 5631/5631 [16:50<00:00,  5.57it/s] 


815 url resolved


...

1040 url resolved


Resolving sub chunk: 100%|██████████| 5631/5631 [14:44<00:00,  6.36it/s]


1004 url resolved


Resolving sub chunk: 100%|██████████| 5631/5631 [13:43<00:00,  6.84it/s]

861 url resolved


In [21]:
chunks_folder = '../../datasets/alexa-top-1m/chunks/2nd_attempt'
resolved_chunks = []

for file in os.listdir(chunks_folder):
  if file.endswith('.csv'):
    resolved_chunks.append(pd.read_csv(f'{chunks_folder}/{file}', index_col=0).reset_index(drop=True))

In [23]:
resolved_df = pd.concat(resolved_chunks)

resolved_df

,url,resolved_url
0,1academy.pro,https://1academy.pro/
1,tuition.io,https://www.tuition.io/
2,kcho.jp,NaN
3,lefigaro.com,NaN
4,trendoffset.com,NaN
...,...,...
16888,lakhimpurlive.com,NaN
16889,fuhaotd.com,NaN
16890,emstock.com.cn,NaN
16891,new-part.com,https://new-part.com/


In [24]:
# resolved VS unresolved URLs
total_rows = resolved_df.shape[0]
f'{(~resolved_df['resolved_url'].isna()).sum()} of {total_rows} resolved VS {resolved_df['resolved_url'].isna().sum()} of {total_rows} unresolved'

'69249 of 371646 resolved VS 302397 of 371646 unresolved'

In [25]:
# resolved VS unresolved URLs percentage
f'{(round((~resolved_df['resolved_url'].isna()).sum() / total_rows * 100))}% of {total_rows} resolved VS {round(resolved_df['resolved_url'].isna().sum() / total_rows * 100)}% of {total_rows} unresolved'

'19% of 371646 resolved VS 81% of 371646 unresolved'

In [28]:
unresolved_df = resolved_df.loc[resolved_df['resolved_url'].isna(), ['url']].reset_index(drop=True)

unresolved_df

,url
0,kcho.jp
1,lefigaro.com
2,trendoffset.com
3,mcit.gov.et
4,h-fg.net
...,...
302392,english4me.net
302393,poderdegym.com
302394,lakhimpurlive.com
302395,fuhaotd.com


In [30]:
for i in range(2, len(unresolved_df)):
  if len(unresolved_df) % i == 0:
    print(f'{i}: {len(unresolved_df) / i}')

3: 100799.0
100799: 3.0


In [38]:
unresolved_df

,url
0,kcho.jp
1,lefigaro.com
2,trendoffset.com
3,mcit.gov.et
4,h-fg.net
...,...
302392,english4me.net
302393,poderdegym.com
302394,lakhimpurlive.com
302395,fuhaotd.com


In [ ]:
chunk_size = len(unresolved_df) // 20
sub_chunk_size = chunk_size // 3

resolve_df_urls(unresolved_df, chunks_with_vpn_folder, chunk_size=chunk_size, sub_chunk_size=sub_chunk_size, max_workers=100)

Processing chunk 11/21 (15119 URLs)


Resolving sub chunk: 100%|██████████| 5039/5039 [06:39<00:00, 12.60it/s]


532 url resolved


Resolving sub chunk: 100%|██████████| 5039/5039 [03:24<00:00, 24.61it/s]


582 url resolved


...

607 url resolved


Resolving sub chunk: 100%|██████████| 2/2 [00:03<00:00,  1.62s/it]


0 url resolved
Processing chunk 21/21 (17 URLs)


Resolving sub chunk: 100%|██████████| 17/17 [00:10<00:00,  1.63it/s]

4 url resolved
